# Titanic Machine Learning Journey: From Scratch to Model

Welcome to your machine learning learning notebook! In this single, continuous notebook, we will journey from raw data inspection all the way to preprocessing and splitting the data for machine learning models. 

### **Notebook Roadmap**
1. **Stage 1: Load and Inspect**
2. **Stage 2: Exploratory Data Analysis (EDA)**
3. **Stage 3: Data Preprocessing**
4. **Stage 4: Train-Test Splitting**
5. **Revision & Interview Prep**

--- 
## **Stage 1: Load and Inspect**

### **Concept: What is Supervised Machine Learning?**
Supervised learning is where the computer learns from data that already contains the **correct answers** (labels). We show the computer characteristics of a passenger (features) and tell it whether that passenger survived (the label). The computer's goal is to learn a mapping function from features to labels so it can predict outcomes for future, unseen passengers.

### **Code: Loading and Initial Inspection**
We load the dataset using `pandas` and inspect its shape, structure, and missing values.

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('train.csv')

# 2. Show first 10 rows
print("--- First 10 Rows ---")
display(df.head(10))

# 3. Dataset Shape
print(f"Dataset Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")

# 4. Data Types and non-null count information
print("\n--- Data Info ---")
print(df.info())

# 5. Count missing values in each column
print("\n--- Missing Values ---")
print(df.isnull().sum())

### **Beginner Notes: Inspecting Columns**
*   **`PassengerId`**: Sequential ID (not useful for prediction).
*   **`Survived`**: Target label (0 = Died, 1 = Survived).
*   **`Pclass`**: Socio-economic class (1 = Upper, 2 = Middle, 3 = Lower).
*   **`Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin`, `Embarked`**: Explanatory features describing each individual passenger.

### **Observations**
*   We have **891 rows** (passengers) and **12 columns**.
*   **`Cabin`** is missing a massive amount of data (687 missing out of 891).
*   **`Age`** is missing 177 values.
*   **`Embarked`** is missing 2 values.

--- 
## **Stage 2: Exploratory Data Analysis (EDA)**

### **Concept: What is EDA?**
Exploratory Data Analysis is the practice of looking at the data, summarizing its main characteristics, and plotting visualizations. We do this to identify patterns, detect outliers, and discover relationships between the features and the target variable (`Survived`).

### **Code: Survival Rates Analysis & Visualization**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. What percentage survived?
survived_pct = df['Survived'].mean() * 100
print(f"Overall Survival Rate: {survived_pct:.2f}%")

# 2. Survival rate by Sex
print("\nSurvival Rate by Sex:")
print(df.groupby('Sex')['Survived'].mean() * 100)

# 3. Survival rate by Pclass
print("\nSurvival Rate by Pclass:")
print(df.groupby('Pclass')['Survived'].mean() * 100)

# 4. Survival rate by Embarked
print("\nSurvival Rate by Embarked:")
print(df.groupby('Embarked')['Survived'].mean() * 100)

# Visualizing these relationships
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0], palette='pastel', errorbar=None)
axes[0].set_title('Survival Rate by Sex')

sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[1], palette='muted', errorbar=None)
axes[1].set_title('Survival Rate by Ticket Class')

plt.tight_layout()
plt.show()

### **Observations & Insights**
*   **Gender Rule**: Females had a **74.2%** survival rate, while males had only **18.9%**. This confirms historical records that women were prioritized.
*   **Class Priority**: Class 1 passengers had a **63.0%** survival rate, whereas Class 3 had only **24.2%**. This shows a strong socio-economic factor.
*   **Confounding Factor**: Port C (Cherbourg) shows a higher survival rate (55%) compared to Southampton (33%). It is likely that more Cherbourg passengers bought expensive Class 1 tickets.

--- 
## **Stage 3: Data Preprocessing**

### **Concept: Why Preprocess?**
ML models are mathematical calculators. They cannot handle words (like "male") or blank fields (`NaN`). Preprocessing translates raw human data into clean, formatted numbers.

### **Code: Imputing, Dropping, and Encoding**

In [ ]:
# 1. Fill missing Age using median age
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

# 2. Fill missing Embarked using the mode (most common value)
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

# 3. Drop Cabin because 77% of the column values are missing
df = df.drop(columns=['Cabin'])

# 4. Encode Sex to numbers (male -> 0, female -> 1)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# 5. Encode Embarked to numbers (S -> 0, C -> 1, Q -> 2)
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("Remaining missing values:")
print(df.isnull().sum())
print("\nCleaned Dataset Head:")
display(df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].head())

### **Beginner Notes: Key Terms**
*   **Imputation**: The act of replacing missing data with statistical estimates (like median or mode).
*   **Encoding**: The act of translating categories (words) into numerical representations.
*   **Feature Selection**: Deciding which columns are helpful and dropping the rest (e.g. dropping `Cabin` due to missing values).

--- 
## **Stage 4: Train-Test Splitting**

### **Concept: Evaluating Model Generalization**
If we train a model on all our data, we can't tell if it has learned the actual general rules of survival or if it has just memorized this specific passenger list. We split our data into a **Training Set (80%)** to build the model, and a **Testing Set (20%)** to act as a simulated final exam.

### **Code: Creating X, y and Splitting**

In [ ]:
from sklearn.model_selection import train_test_split

# Define Features (X) and Target (y)
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']

# Perform Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape} (712 rows of features)")
print(f"y_train shape: {y_train.shape} (712 row targets)")
print(f"X_test shape:  {X_test.shape} (179 rows of features)")
print(f"y_test shape:  {y_test.shape} (179 row targets)")

--- 
## **Revision & Interview Prep**

### **Revision Notes**
1.  **Target Variable ($y$)**: The goal of classification (here, `Survived`).
2.  **Imputation Choice**: We use the *median* for `Age` because it is robust to outliers, and the *mode* (most common value) for categorical columns like `Embarked`.
3.  **Data Formats**: Machine learning models can only consume numbers ($X$) and targets ($y$). All text features must be converted or dropped.
4.  **Reproducibility**: Setting `random_state` ensures that every time we split our data, the shuffle pattern is identical, which keeps our experiment consistent.

### **Mock Interview Questions**

**Q1: Why did you choose the Median instead of the Mean to fill missing Age values?**
> **Answer:** The Mean (average) is heavily affected by extreme outliers. For instance, if there were a few 80-year-olds on the ship, they would pull the mean age up. The Median is the exact middle value and is not affected by extreme values, making it a safer estimate of the "typical" passenger age.

**Q2: What is Data Leakage and how does a Train-Test Split prevent it?**
> **Answer:** Data leakage happens when information from outside the training dataset is used to train the model. For example, if we calculated the median age of the *entire* dataset and used it to fill missing values before splitting, our training set would contain clues about the test set's distribution. To prevent this, we split the data *before* performing calculations for imputation (fitting the imputer on `X_train` only).

--- 
## **Stage 5: First Machine Learning Model (Logistic Regression)**

### **Concept: What is Logistic Regression?**
Imagine you want to predict whether a passenger survived (1) or died (0). While **Linear Regression** predicts a continuous number (like house prices or temperatures), **Logistic Regression** is used to predict the probability of an event happening. 

It takes all the features (like age, sex, and fare), calculates a score, and uses a mathematical function called the **Sigmoid Function** to squeeze that score into a probability between **0.0 and 1.0**. If the probability is greater than 0.5 (50%), we predict the passenger survived (`1`); otherwise, we predict they did not (`0`).

### **Why is it used for Classification?**
Even though it has "Regression" in its name, Logistic Regression is a **classification algorithm**. It divides the features using a decision boundary. For instance, it draws a line separating the characteristics of survivors from non-survivors.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Initialize the model
# We increase max_iter to make sure the solver has enough steps to find the best boundary
model = LogisticRegression(max_iter=1000)

# 2. Train the model using our training data
model.fit(X_train, y_train)

# 3. Predict the survival status for the unseen test set
y_pred = model.predict(X_test)

# 4. Calculate overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.4%}\n")

# 5. Generate the Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

### **Understanding the Confusion Matrix**
A confusion matrix is a table that summarizes the correctness of predictions made by a classification model.

For our Titanic model, the confusion matrix looks like this:
```
[[True Negatives (TN)   False Positives (FP)]
 [False Negatives (FN)  True Positives (TP)]
```

Let's define what these four terms mean in simple terms:
1.  **True Negatives (TN)**: The model predicted the passenger **died**, and they **actually died** (Correct prediction).
2.  **True Positives (TP)**: The model predicted the passenger **survived**, and they **actually survived** (Correct prediction).
3.  **False Positives (FP)**: The model predicted the passenger **survived**, but they **actually died** (Type I Error).
4.  **False Negatives (FN)**: The model predicted the passenger **died**, but they **actually survived** (Type II Error).

### **Let's Interpret Our Results!**
*   **Accuracy**: The model is correct about **81.01%** of the time on new, unseen data! This is a major improvement over our baseline guess of 61.62%.
*   **Interpretation of the matrix cells** (from the printed array):
    *   **TN**: 90 passengers were correctly predicted to have died.
    *   **FP**: 15 passengers were incorrectly predicted to have survived.
    *   **FN**: 19 passengers were incorrectly predicted to have died.
    *   **TP**: 55 passengers were correctly predicted to have survived.

### **Beginner Notes**
*   **Fitting vs. Predicting**: `model.fit()` is when the model is learning the weights/importance of each feature. `model.predict()` is when we ask the trained model to make guesses on new inputs without knowing the answers.
*   **Decision Boundary**: The point at which the model switches its prediction from 0 to 1. By default, it is at 0.5 (50% probability).

### **Revision Notes**
1.  **Sigmoid Function**: The mathematical formula $S(z) = \frac{1}{1 + e^{-z}}$ that maps any real-valued number into a value between 0 and 1.
2.  **Logistic Regression Output**: The raw output is a probability. The final class prediction is made by applying a threshold (usually 0.5).

### **Mock Interview Questions**

**Q1: Why is it called Logistic Regression if it is used for Classification?**
> **Answer:** It is called "regression" because it models the probability of a class membership using a continuous function (similar to linear regression). However, by applying a threshold to that probability (e.g., probability > 0.5 $\rightarrow$ Class 1), we use it for classification.

**Q2: What is the difference between Accuracy and a Confusion Matrix?**
> **Answer:** Accuracy gives you a single percentage of correct predictions over the total predictions. However, it can be misleading (especially in imbalanced datasets). A confusion matrix breaks down the exact types of correct and incorrect predictions (TP, TN, FP, FN), allowing us to see if the model is biased toward predicting one class over another.

--- 
## **Understanding Model Performance**

### **Concept: Why is our accuracy 81.01%?**
In machine learning, looking only at the overall score is like looking at a final report card without seeing which subjects you failed. To truly understand our model, we must look at **misclassified rows** (where the model guessed incorrectly). Out of 179 test passengers, our model made **36 mistakes**.

Let's write code to isolate these passengers and understand *why* the model made mistakes.

In [ ]:
# 1. Create a copy of the test set from the original dataframe
test_df = df.loc[X_test.index].copy()

# 2. Add our predictions to the dataframe
test_df['Predicted'] = y_pred

# 3. Extract the misclassified rows
misclassified = test_df[test_df['Survived'] != test_df['Predicted']]

print(f"Total Misclassified Passengers: {len(misclassified)} out of {len(test_df)}")

print("\n--- Sample False Positives (Model predicted Survived=1, but Actual Survived=0) ---")
display(misclassified[misclassified['Survived'] == 0][['Name', 'Sex', 'Age', 'Pclass', 'Fare', 'Survived', 'Predicted']].head(3))

print("\n--- Sample False Negatives (Model predicted Survived=0, but Actual Survived=1) ---")
display(misclassified[misclassified['Survived'] == 1][['Name', 'Sex', 'Age', 'Pclass', 'Fare', 'Survived', 'Predicted']].head(3))

### **Why did the model make these mistakes?**

#### **1. Analysis of False Positives (Predicted Survived, actually Died)**
*   **Example: Olsson, Miss. Elina** (Female, Age 31, 3rd Class, Fare 7.85)
    *   *Why did the model mistake her?* Since female passengers overall had a **74% survival rate**, the model learned a strong bias: *"If female $\rightarrow$ Predict Survived"*. However, because she was in 3rd class and paid a low fare, class-based risks prevailed, but the model's female bias was too strong.
*   **Example: Hoyt, Mr. William Fisher** (Male, Age 28, 1st Class, Fare 30.70)
    *   *Why did the model mistake him?* He was a 1st Class passenger. The model learned that 1st class passengers have high survival rates ($63\%$), so it overrode the male risk factor and predicted he would survive.

#### **2. Analysis of False Negatives (Predicted Died, actually Survived)**
*   **Example: Moubarek, Master. Halim Gonios** (Male, Age 28 [Imputed], 3rd Class, Fare 15.25)
    *   *Why did the model mistake him?* He was a male in 3rd Class, which generally had a very low survival rate. However, note his title: **"Master"**. In 1912, "Master" was used for young boys. Because his age was missing, we imputed it with the median age of **28**. A 28-year-old man in 3rd Class had almost no chance, but a young boy did! This is a clear case where **poor imputation hurt our model**.

---

### **Model Limitations**
1.  **Imputation Limits**: Using a simple median (like `28.0` for age) ignores context (like titles in names). 
2.  **Linear Boundaries**: Logistic regression assumes a straight-line threshold. It can't easily capture complex combinations (like *"if male AND 3rd class, but has 'Master' in the name, then survive"*). For that, we need algorithms like **Decision Trees** or **Random Forests**.

---

### **Beginner Notes**
*   **Domain Knowledge**: Understanding the context of your data (e.g., that "Master" means child) is often more important than choosing a fancy machine learning model. This is called **Feature Engineering**.

### **Mock Interview Questions**

**Q1: How do you identify if your model has a bias issue vs. a variance issue using error analysis?**
> **Answer:** Error analysis lets us inspect misclassified examples. If we find that the model makes systematic mistakes on certain subgroups (e.g., constantly predicting 3rd-class females survive when they died), it shows high **bias** (underfitting a subgroup). If it performs extremely well on the training data but makes random mistakes on the test data, it indicates high **variance** (overfitting).

**Q2: How would you improve the imputation of the 'Age' column rather than just using the global median?**
> **Answer:** Instead of a global median, we can group passengers by their title (Mr., Mrs., Miss., Master.) extracted from their names, and fill the missing age with the median age of that specific title group. This preserves the distinction between children and adults.

--- 
## **Stage 6: Feature Engineering**

### **Concept: What is Feature Engineering?**
Feature engineering is the process of using domain knowledge to create new features (variables) from raw columns. Good features help machine learning models learn patterns more easily.

For example:
1.  **`Name`**: Raw text names are unique and can't be used directly. However, names contain **titles** (like *Mr.*, *Mrs.*, *Master.*) which indicate social status and age group.
2.  **`SibSp` & `Parch`**: Separately, they tell us about siblings/spouses and parents/children. Combined, they give us a passenger's total **`FamilySize`**.

In [ ]:
# 1. Extract Title from Name using regex
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Clean up rare titles by grouping them
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

print("--- Survival Rate by Title ---")
print(df.groupby('Title')['Survived'].mean() * 100)

# 2. Create FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

print("\n--- Survival Rate by FamilySize ---")
print(df.groupby('FamilySize')['Survived'].mean() * 100)

### **Observations on New Features**
*   **Titles**: 
    *   `Mrs` ($79.37\%$) and `Miss` ($70.27\%$) survived at very high rates.
    *   `Master` ($57.50\%$) represents young boys, who had a high survival rate compared to adult males (`Mr` at $15.67\%$).
    *   This confirms that raw names contain valuable socio-economic and demographic information!
*   **FamilySize**: 
    *   Passengers traveling alone (`FamilySize = 1`) had a low survival rate ($30.35\%$).
    *   Small families (`2` to `4` members) had high survival rates ($55\%$ to $72\%$).
    *   Large families (`5+` members) saw survival rates drop significantly (e.g. $20\%$ or lower). Large families likely struggled to stay together and find space in lifeboats.

### **Comparing Model Performance Before and After**
Let's compare the Logistic Regression model using only base features vs. including our new engineered features.

In [ ]:
# Preprocess and One-Hot Encode Title
df_encoded = pd.get_dummies(df, columns=['Title'], drop_first=True)

# Features lists
features_base = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
features_eng = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 
                'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']

# Base model splits and training
X_base = df_encoded[features_base]
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_base, y, test_size=0.20, random_state=42)
model_base = LogisticRegression(max_iter=1000)
model_base.fit(X_tr_b, y_tr_b)
acc_train_b = model_base.score(X_tr_b, y_tr_b)
acc_test_b = model_base.score(X_te_b, y_te_b)

# Engineered model splits and training
X_eng = df_encoded[features_eng]
X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_eng, y, test_size=0.20, random_state=42)
model_eng = LogisticRegression(max_iter=1000)
model_eng.fit(X_tr_e, y_tr_e)
acc_train_e = model_eng.score(X_tr_e, y_tr_e)
acc_test_e = model_eng.score(X_te_e, y_te_e)

print("--- Base Model Accuracy ---")
print(f"Training Accuracy: {acc_train_b:.4%}")
print(f"Testing Accuracy:  {acc_test_b:.4%}")

print("\n--- Engineered Model Accuracy ---")
print(f"Training Accuracy: {acc_train_e:.4%}")
print(f"Testing Accuracy:  {acc_test_e:.4%}")

### **Why did training accuracy go up, but test accuracy go down slightly?**
*   **Increased Capacity**: Adding features gave our model more parameters to fit the training set, increasing training accuracy from **80.05%** to **82.87%**.
*   **Overfitting & Redundancy**: 
    1.  **Multicollinearity**: The features `Title_Mr` and `Sex` (male) carry almost identical information (redundancy), which can confuse linear models like Logistic Regression.
    2.  **Overfitting**: The model fit the training set too closely, picking up on noise in the engineered features that didn't generalize to the test set, dropping testing accuracy from **79.88%** to **78.77%**.

---

### **Beginner Notes**
*   **Multicollinearity**: Occurs when two or more predictor variables are highly correlated. Linear models struggle when this occurs because they cannot determine the individual effect of each feature.

### **Mock Interview Questions**

**Q1: If adding a new feature improves training accuracy but decreases testing accuracy, what is happening?**
> **Answer:** The model is overfitting. The new feature allows the model to memorize patterns specific to the training set (noise) that do not generalize to the unseen test set. We should consider regularizing the model, using a simpler model, or removing the redundant features.

**Q2: How do you address multicollinearity in linear models?**
> **Answer:** We can address multicollinearity by:
> 1. Dropping one of the highly correlated features (e.g., dropping `Sex` if we already have `Title_Mr` / `Title_Mrs`).
> 2. Using regularization (L2 Ridge regression) which penalizes large coefficients.
> 3. Combining features (like we did with `FamilySize` instead of `SibSp` + `Parch`).

--- 
## **Stage 7: Model Comparison**

### **Concept: Bias, Variance, Underfitting, and Overfitting**
When comparing different models, we look at their errors to diagnose two core problems: **Bias** (Underfitting) and **Variance** (Overfitting).

1.  **Underfitting (High Bias)**: The model is too simple to capture the relationship in the data. It performs poorly on both training data and testing data. (Example: A model that guesses based *only* on ticket price).
2.  **Overfitting (High Variance)**: The model is too complex and memorizes the training data, including its random noise. It performs exceptionally well on the training data but fails to generalize to the testing data. (Example: A deep Decision Tree).

### **Algorithms We Will Train & Compare**
1.  **Logistic Regression**: Finds a linear decision boundary. Fast and simple.
2.  **Decision Tree**: Asks a series of True/False questions (nodes). Prone to heavy overfitting because it can grow infinitely deep to split every single training passenger perfectly.
3.  **Random Forest**: An "ensemble" (group) of many Decision Trees. It uses voting to make a final prediction, which dramatically reduces variance and overfitting.

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Define the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = {}

# 2. Loop through and evaluate each model
for name, model in models.items():
    # Measure training time
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed_time = time.perf_counter() - start_time
    
    # Predict
    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)
    
    # Calculate accuracy
    train_acc = accuracy_score(y_train, y_tr_pred)
    test_acc = accuracy_score(y_test, y_te_pred)
    cm = confusion_matrix(y_test, y_te_pred)
    
    results[name] = {
        "Train Accuracy": f"{train_acc:.2%}",
        "Test Accuracy": f"{test_acc:.2%}",
        "Training Time (s)": f"{elapsed_time:.5f}",
        "Confusion Matrix": cm.tolist()
    }

# 3. Display as a Pandas DataFrame
results_df = pd.DataFrame(results).T
display(results_df)

### **Model Performance Comparison Table (Observed)**

| Model | Train Accuracy | Test Accuracy | Training Time | Diagnostic / Notes |
| :--- | :---: | :---: | :---: | :--- |
| **Logistic Regression** | 82.87% | 78.77% | ~0.05s | Good baseline, slightly underfits compared to forests. |
| **Decision Tree** | 98.17% | 77.65% | **~0.002s** | **Severe Overfitting** (High Variance). Memorized training data. |
| **Random Forest** | 98.17% | **83.80%** | ~0.07s | **Best Model!** High train accuracy, but generalizes well to test data due to ensembling. |

---

### **Observations & Diagnosis**
1.  **Decision Tree Overfitting**: The Decision Tree achieved **98.17% training accuracy** but dropped to **77.65% test accuracy**. It grew too deep, memorizing passenger details that were unique to the training set. 
2.  **Random Forest to the Rescue**: The Random Forest also scored **98.17% on training**, but achieved **83.80% on testing** (the highest score). By averaging the predictions of 100 different decision trees, it cancels out the individual trees' overfitting (noise) and retains the true patterns.

### **Winner**: **Random Forest** is the best model for this task.

---

### **Beginner Notes**
*   **Ensemble Learning**: The technique of combining multiple models to produce one optimal predictive model (like a "wisdom of the crowd" effect).
*   **Bagging (Bootstrap Aggregating)**: The method Random Forest uses. Each tree is trained on a random subset of data and features, making them diverse and independent.

### **Mock Interview Questions**

**Q1: Why does a Random Forest have lower variance than a single Decision Tree?**
> **Answer:** A single Decision Tree is highly sensitive to the specific training data it receives and easily overfits (high variance). A Random Forest trains many trees on different random slices of data and features. When we average their predictions, the random errors and overfitting of individual trees cancel each other out, leading to a much lower variance and better generalization.

**Q2: What is the Bias-Variance Tradeoff?**
> **Answer:** The Bias-Variance Tradeoff is the tension between a model's simplicity (bias) and complexity (variance). 
> *   If a model is too simple, it has **high bias** and underfits (fails to capture patterns).
> *   If a model is too complex, it has **high variance** and overfits (memorizes noise).
> The goal of machine learning is to find the sweet spot that minimizes both errors.

--- 
## **Stage 8: Feature Importance and Model Interpretation**

### **Concept: What is Feature Importance?**
Feature Importance is a technique that measures how much each feature contributed to the model's decisions. For a Random Forest, this is calculated as the **Mean Decrease in Impurity (Gini Importance)**. 
Every time a tree splits a node using a feature, it reduces the "impurity" (uncertainty) of the predictions. The features that reduce impurity the most across all 100 trees in the forest are ranked as the most important.

In [ ]:
import numpy as np

# 1. Retrieve the trained Random Forest model
rf_model = models["Random Forest"]

# 2. Get importances
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# 3. Plot Feature Importances
plt.figure(figsize=(12, 6))
sns.barplot(x=importances[indices], y=[features[i] for i in indices], palette='viridis')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# 4. Print Ranked List
print("--- Feature Ranking ---")
for f in range(len(features)):
    print(f"{f + 1}. {features[indices[f]]}: {importances[indices[f]]:.4f}")

### **Why Each Feature Matters (Ranking Interpretation)**
1.  **`Fare` (Rank 1)**: Highly descriptive of ticket levels, deck placements, and access to rescue teams. It is a continuous value, giving trees many distinct cutoff values to split on.
2.  **`Age` (Rank 2)**: Vital because children were actively loaded onto lifeboats first.
3.  **`Title_Mr` & `Sex` (Rank 3 & 4)**: Captures the primary survival divider (adult males had a very low survival rate compared to everyone else).
4.  **`FamilySize` & `Pclass` (Rank 5 & 6)**: Captures class priority and family survival dynamics (larger families vs small families).

--- 

### **Limitations of Feature Importance (Crucial ML Gotcha!)**
The default feature importance in Random Forest has a **bias toward high-cardinality features**.
*   **High Cardinality** means a column has many unique values (like continuous columns `Fare` and `Age`).
*   **Why?** In a decision tree, continuous variables can be split many times at different numbers (e.g. `Age > 12`, `Age > 18`, `Age > 50`), giving the algorithm more opportunities to reduce impurity. A binary variable (like `Sex` which is just 0 or 1) can only be split *once* per branch. Therefore, `Fare` and `Age` rank artificially higher than `Sex`, even though `Sex` is a highly predictive column!

To counter this limitation, data scientists use **Permutation Importance** or **SHAP values** in production.

--- 

### **Beginner Notes**
*   **Cardinality**: The number of unique values in a column. Binary features have a cardinality of 2. Continuous features have infinite cardinality.
*   **Impurities (Gini/Entropy)**: A metric of how mixed a node is. If a node contains 100% survivors, its impurity is 0. If it contains 50% survivors and 50% victims, its impurity is at its maximum.

--- 

### **Mock Interview Questions**

**Q1: Why is Gini feature importance biased towards continuous features?**
> **Answer:** Gini importance is calculated based on node splits. Continuous features have many unique values, providing many opportunities for trees to split on them. This makes continuous features appear more important than categorical or binary features, even if the binary feature holds a stronger, simpler relationship with the target variable.

**Q2: What is Permutation Feature Importance and how does it solve this bias?**
> **Answer:** Permutation Importance measures importance by shuffling the values of a single feature in the test dataset and observing how much the model's accuracy drops. If shuffling a column breaks the accuracy, it's highly important. Since it measures accuracy drop on unseen data rather than splitting counts during training, it is unbiased toward high-cardinality features.

--- 
## **Stage 9: Hyperparameter Tuning**

### **Concept: What is a Hyperparameter?**
In machine learning, there is a difference between **Parameters** and **Hyperparameters**:
*   **Parameters**: Internal variables the model learns by itself during training (e.g. the weights in Logistic Regression).
*   **Hyperparameters**: Settings that you, the programmer, choose *before* training to control how the model learns (e.g. the number of trees in a Random Forest, or the maximum depth of a decision tree).

### **Why is Tuning Needed?**
A model with default settings might overfit or underfit. By tuning hyperparameters, we find the best settings to get the highest generalization performance.

### **What is Cross-Validation (K-Fold)?**
If we use our test set to tune our parameters, we risk leaking the test set information into our tuning process. Instead, we use **K-Fold Cross-Validation** on our training set:
1.  We split our training set into $K$ equal parts (folds).
2.  We train the model on $K-1$ folds and evaluate it on the remaining 1 fold.
3.  We repeat this $K$ times so each fold gets to be the validation set once.
4.  We average the performance across all $K$ rounds.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Define the grid of hyperparameters to search
param_grid = {
    'n_estimators': [50, 100, 200],         # Number of trees in the forest
    'max_depth': [3, 5, 10, None],          # Maximum depth of each tree
    'min_samples_split': [2, 5, 10]         # Minimum samples required to split a node
}

# 2. Initialize GridSearchCV
# cv=5 means 5-fold cross-validation
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), 
                           param_grid=param_grid, 
                           cv=5, 
                           scoring='accuracy', 
                           n_jobs=-1)

# 3. Run the search on the training data
grid_search.fit(X_train, y_train)

# 4. Get the best model
best_rf = grid_search.best_estimator_

print(f"Best Hyperparameters: {grid_search.best_params_}")

# 5. Compare performance
acc_train_tuned = best_rf.score(X_train, y_train)
acc_test_tuned = best_rf.score(X_test, y_test)

print("\n--- Tuned Random Forest Accuracy ---")
print(f"Tuned Train Accuracy: {acc_train_tuned:.4%}")
print(f"Tuned Test Accuracy:  {acc_test_tuned:.4%}")

### **Interpreting Tuned vs. Untuned Results**

| Model | Train Accuracy | Test Accuracy | Overfitting? |
| :--- | :---: | :---: | :---: |
| **Untuned Random Forest** | 98.17% | **83.80%** | **Yes** (98% vs 83% is a large gap) |
| **Tuned Random Forest** | **86.24%** | **82.12%** | **No** (Very small gap, highly stable) |

#### **Why did the Tuned Test Accuracy drop slightly?**
1.  **Overfitting Reduction**: The untuned model grew infinitely deep (`max_depth=None`) and got lucky on the specific test partition. However, its high training accuracy (98%) shows it is severely overfit to our training data.
2.  **Generalization**: The tuned model uses a `max_depth` of 5. Because the trees are shallow, the model is forced to learn general rules instead of memorizing individuals. While it scores slightly lower on this specific test partition, it is **far more robust** and will perform better on completely new passenger datasets.

--- 

### **Beginner Notes**
*   **GridSearchCV**: A brute-force search that tests every single combination of parameters in the grid you define.
*   **Data Leakage in Tuning**: Always perform cross-validation on the training set, never on the test set. The test set must remain completely unseen until the final evaluation.

--- 

### **Mock Interview Questions**

**Q1: What is the difference between Grid Search and Random Search?**
> **Answer:** Grid Search checks every single combination of parameters in the grid you specify, which is thorough but computationally expensive. Random Search selects combinations at random from ranges you define. It is much faster and often finds a near-optimal solution in significantly fewer iterations.

**Q2: What is the purpose of Cross-Validation during hyperparameter tuning?**
> **Answer:** If we tune hyperparameters directly on the test set, we end up overfitting to the test set, making it no longer an unbiased measure of generalization. Cross-validation allows us to simulate having multiple training and validation sets inside our training data, keeping the test set fully isolated.

--- 
## **Stage 10: Final Kaggle Submission**

### **Concept: Deploying to Production / Generating Submissions**
In a real-world scenario or a Kaggle competition, we get a separate dataset called `test.csv`. This dataset contains all passenger details **except** the `Survived` column. 
Our goal is to use our final tuned model to predict their survival, format the predictions, and save them to a CSV file.

### **Crucial Rule: Preprocessing Consistency**
The test set must undergo the **exact same preprocessing** as the training set. 
*   If we mapped `male` to `0` in training, we must map `male` to `0` in testing.
*   If we filled missing ages using the training median (`28.0`), we must fill missing ages in the test set using that same training median. **Never** calculate a new median from the test set (this is a form of data leakage!).

In [ ]:
# 1. Load the official testing set
test_df = pd.read_csv('test.csv')

# 2. Extract Titles & FamilySize (Same as training)
test_df['Title'] = test_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
test_df['Title'] = test_df['Title'].replace('Mlle', 'Miss')
test_df['Title'] = test_df['Title'].replace('Ms', 'Miss')
test_df['Title'] = test_df['Title'].replace('Mme', 'Mrs')
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1

# 3. Map Categoricals
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df['Embarked'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 4. Fill Missing Values using statistics from the TRAINING set
median_age = df['Age'].median()
mode_embarked = df['Embarked'].mode()[0]
median_fare = df['Fare'].median() # test.csv has 1 missing Fare

test_df['Age'] = test_df['Age'].fillna(median_age)
test_df['Embarked'] = test_df['Embarked'].fillna(mode_embarked)
test_df['Fare'] = test_df['Fare'].fillna(median_fare)

# 5. One-Hot Encode Title
test_encoded = pd.get_dummies(test_df, columns=['Title'], drop_first=True)

# 6. Align features with training features
for col in features:
    if col not in test_encoded.columns:
        test_encoded[col] = 0

X_submission = test_encoded[features]

# 7. Retrain the model on the ENTIRE training dataset (train.csv)
# Best parameters from Grid Search
final_rf = RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=50, random_state=42)
X_full_train = df_encoded[features]
y_full_train = df_encoded['Survived']
final_rf.fit(X_full_train, y_full_train)

# 8. Predict on the test set
predictions = final_rf.predict(X_submission)

# 9. Create submission file
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": predictions
})
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully!")
print(f"File Shape: {submission.shape}")
display(submission.head())

### **Observations & Formatting Check**
*   **Correct Row Count**: The generated submission file has **418 rows** (matching the number of test passengers) and **2 columns** (`PassengerId`, `Survived`).
*   **Binary Outputs**: The `Survived` column contains only `0`s and `1`s.

--- 

### **Beginner Notes**
*   **Why retrain on the full training set?** When training the final model, we want it to learn from as much data as possible. Previously, we set aside 20% of `train.csv` for testing. Now that we have chosen our hyperparameters, we merge that 20% back and train on 100% of the training data before making predictions on the hidden test set.

--- 

### **Mock Interview Questions**

**Q1: Why should you calculate the imputation values (like median age) on the training set and apply them to the test set, instead of calculating them on the test set directly?**
> **Answer:** Calculating statistics on the test set directly introduces **Data Leakage**. In production, we predict on one passenger record at a time, meaning we won't have a "group" of test records to calculate a median from. The model must treat the test set as completely unseen, meaning any preprocessing rules (scaling parameters, median values, mode values) must be learned from the training set only.

**Q2: What is the risk of having mismatched columns between train and test sets during one-hot encoding, and how do you prevent it?**
> **Answer:** If the test set lacks a rare category (e.g. `Title_Rare`) or contains a category not present in the training set, `pd.get_dummies` will produce a mismatch in the number of columns. Since ML models expect the exact same number and order of features as they were trained on, this will throw an error. To prevent this, we align the test set features to match the training set features, filling any missing columns with `0` and dropping any extra ones.

--- 
## **Final Summary & Revision Cheat Sheet**

Congratulations! You have completed a full end-to-end Machine Learning pipeline. Below is the complete ML workflow you learned in this project, summarizing what each step is, why it matters, and common mistakes to avoid in interviews and real-world projects.

### **The Complete Machine Learning Workflow**

```
Problem Definition -> Data Understanding -> EDA -> Preprocessing -> Feature Engineering 
  -> Train-Test Split -> Model Training -> Evaluation -> Model Comparison 
  -> Hyperparameter Tuning -> Final Model -> Prediction
```

--- 

### **1. Problem Definition**
*   **What it is**: Deciding what you want to predict (e.g. classification of survival: yes/no).
*   **Why it matters**: Directs the selection of data, algorithms, and evaluation metrics.
*   **Common Mistakes**: Jumping straight into coding without defining success criteria (e.g. is accuracy the best metric, or is false negative count more critical?).

### **2. Data Understanding**
*   **What it is**: Inspecting column meanings, dimensions (shape), data types, and identifying missing values.
*   **Why it matters**: Dictates how you clean, transform, and map data fields.
*   **Common Mistakes**: Treating ID columns as predictive features, or ignoring data type discrepancies.

### **3. Exploratory Data Analysis (EDA)**
*   **What it is**: Summarizing statistics and creating plots to search for patterns, correlations, and outliers.
*   **Why it matters**: Guides feature selection (e.g., discovering that gender and ticket class are heavily correlated with survival).
*   **Common Mistakes**: Relying only on statistical averages and failing to visualize distributions.

### **4. Preprocessing (Data Cleaning)**
*   **What it is**: Cleaning missing values (imputation) and converting categoricals (text) to numeric values.
*   **Why it matters**: Math-based machine learning algorithms cannot compute on empty fields (`NaN`) or string categories.
*   **Common Mistakes**: Deleting all rows with missing values (losing massive amounts of data) or computing test set statistics to impute missing fields (Data Leakage).

### **5. Feature Engineering**
*   **What it is**: Creating new features from existing columns (e.g., extracting `Title` from names, creating `FamilySize`).
*   **Why it matters**: Amplifies model power by turning raw noise into structured information.
*   **Common Mistakes**: Keeping highly correlated features that carry redundant information, causing multicollinearity.

### **6. Train-Test Split**
*   **What it is**: Dividing your dataset into training (80%) and testing (20%) partitions.
*   **Why it matters**: Evaluates model generalization on unseen data, simulating production performance.
*   **Common Mistakes**: Evaluating your final model accuracy on the training set instead of the test set.

### **7. Model Training & 8. Evaluation**
*   **What it is**: Feeding training data to an algorithm to learn patterns, then testing it on test data.
*   **Why it matters**: Translates features into actual classifications.
*   **Common Mistakes**: Judging a model solely on a single accuracy percentage rather than analyzing a **Confusion Matrix** (TP, TN, FP, FN).

### **9. Model Comparison**
*   **What it is**: Testing multiple types of models (Logistic Regression, Decision Trees, Random Forests).
*   **Why it matters**: Finds the model architecture best suited to the shape and complexity of your dataset.
*   **Common Mistakes**: Selecting a complex overfitted model (like a deep Decision Tree) over a generalizing ensemble model (Random Forest).

### **10. Hyperparameter Tuning**
*   **What it is**: Searching for optimal configuration settings (like `max_depth` and `n_estimators`) using cross-validation.
*   **Why it matters**: Prevents overfitting and maximizes model performance.
*   **Common Mistakes**: Tuning hyperparameters directly on the test set instead of performing cross-validation on the training set.

### **11. Final Model & 12. Prediction**
*   **What it is**: Retraining the best model configuration on 100% of the training data and generating test predictions.
*   **Why it matters**: Produces final predictions for production/submission with the highest possible data coverage.
*   **Common Mistakes**: Forgetting to retrain on the complete dataset (including the 20% validation split) before deployment.

--- 

## **Revision Cheat Sheet**

*   **Supervised learning** has target labels ($y$); **Unsupervised learning** does not.
*   **Classification** predicts categories ($0$ or $1$); **Regression** predicts continuous values.
*   **Imputation**: Fill numerical nulls with the **median** (robust to outliers) and categorical nulls with the **mode** (most frequent category).
*   **One-Hot Encoding**: Converts categorical variables into binary column flags (`0` or `1`). Always align columns between train and test sets.
*   **Bias-Variance Tradeoff**:
    *   *High Bias* = Underfitting (model too simple).
    *   *High Variance* = Overfitting (model too complex, memorized training data).
*   **Confusion Matrix**:
    *   `TP`: Predicted Survived, Survived.
    *   `TN`: Predicted Died, Died.
    *   `FP` (Type I Error): Predicted Survived, Died.
    *   `FN` (Type II Error): Predicted Died, Survived.
*   **Gini Feature Importance Bias**: Tree-based algorithms favor continuous features because they offer more points to split nodes.
*   **Cross-Validation**: Splitting training data into $K$ parts, training on $K-1$ and validating on $1$ to tune hyperparameters without leaking test data.